In [1]:
import json
import pandas as pd
from abc import ABC, abstractmethod


# ============================================================
# CONFIG
# ============================================================

DATA_PATH = "D:/FM/ALGOTRADING/artifacts/backtest/hist_data_D.json"

OUTPUT_PATH = (
    "D:/FM/ALGOTRADING/artifacts/backtest/"
    "backtest_trades.csv"
)

CLOSE_THRESHOLD = 0.70
TARGET_PERCENT = 0.10


# ============================================================
# EXIT RESULT
# ============================================================

class ExitResult:

    def __init__(
        self,
        exit_price,
        exit_reason
    ):
        self.exit_price = exit_price
        self.exit_reason = exit_reason

    def to_dict(self):
        return {
            "exit_price": self.exit_price,
            "exit_reason": self.exit_reason
        }


# ============================================================
# BASE EXIT RULE
# ============================================================

class ExitRule(ABC):

    @abstractmethod
    def check(self, row, entry_price, sl, target):
        """
        Check whether this exit condition is triggered.

        Returns:
            ExitResult
            or None
        """
        pass


# ============================================================
# STOP LOSS EXIT
# ============================================================

class SLExit(ExitRule):

    def check(
        self,
        row,
        entry_price,
        sl,
        target
    ):

        if row["low"] <= sl:

            return ExitResult(
                exit_price=sl,
                exit_reason="SL"
            )

        return None


# ============================================================
# TARGET EXIT
# ============================================================

class TargetExit(ExitRule):

    def check(
        self,
        row,
        entry_price,
        sl,
        target
    ):

        if row["high"] >= target:

            return ExitResult(
                exit_price=target,
                exit_reason="TARGET"
            )

        return None


# ============================================================
# EMA EXIT
# ============================================================

class EMAExit(ExitRule):

    def check(
        self,
        row,
        entry_price,
        sl,
        target
    ):

        if row["close"] < row["9_ema"]:

            return ExitResult(
                exit_price=row["close"],
                exit_reason="9_EMA"
            )

        return None


# ============================================================
# EXIT ENGINE
# ============================================================

class ExitEngine:

    def __init__(self, rules):

        if not rules:
            raise ValueError(
                "At least one exit rule is required."
            )

        self.rules = rules

    def check(
        self,
        row,
        entry_price,
        sl,
        target
    ):
        """
        Check all configured exit rules.

        The FIRST rule that triggers wins.

        Therefore, the order of rules determines
        exit priority.
        """

        for rule in self.rules:

            result = rule.check(
                row=row,
                entry_price=entry_price,
                sl=sl,
                target=target
            )

            if result is not None:
                return result

        return None


# ============================================================
# EXIT SCENARIO
# ============================================================
#
# CHANGE ONLY THIS SECTION WHEN YOU WANT TO
# TEST A DIFFERENT EXIT STRATEGY.
#
# IMPORTANT:
# Rule order = priority.
#
# ============================================================

exit_engine = ExitEngine(
    rules=[
        TargetExit(),
        SLExit(),
        EMAExit(),
    ]
)


# ============================================================
# LOAD DATA
# ============================================================

with open(DATA_PATH, "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)


# ============================================================
# VALIDATE INPUT DATA
# ============================================================

required_columns = [
    "symbol",
    "date",
    "open",
    "high",
    "low",
    "close"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:

    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ============================================================
# PREPARE DATA
# ============================================================

df["date"] = pd.to_datetime(df["date"])

df = (
    df.sort_values(
        ["symbol", "date"]
    )
    .reset_index(drop=True)
)


# ============================================================
# PREVIOUS DAY VALUES
# ============================================================

df["prev_day_close"] = (
    df.groupby("symbol")["close"]
      .shift(1)
)

df["prev_day_high"] = (
    df.groupby("symbol")["high"]
      .shift(1)
)

df["prev_day_low"] = (
    df.groupby("symbol")["low"]
      .shift(1)
)


# ============================================================
# DAILY RANGE %
# ============================================================

df["range"] = (
    (df["high"] - df["low"]) /
    df["low"]
) * 100


# ============================================================
# 9 EMA
# ============================================================

df["9_ema"] = (
    df.groupby("symbol")["close"]
      .transform(
          lambda x: x.ewm(
              span=9,
              adjust=False
          ).mean()
      )
      .round(2)
)


# ============================================================
# CLOSE POSITION INSIDE CANDLE
# ============================================================

df["close_>_70th_range"] = (
    df["close"] >=
    (
        df["low"] +
        (
            df["high"] - df["low"]
        ) * CLOSE_THRESHOLD
    )
)


# ============================================================
# TRADE SIGNAL
# ============================================================

df["trade_signal"] = (
    (df["close"] > df["prev_day_close"]) &
    (df["high"] > df["prev_day_high"]) &
    (df["low"] > df["prev_day_low"]) &
    (df["close_>_70th_range"])
)


# ============================================================
# FIND EXIT
# ============================================================

def find_exit(
    symbol_df,
    entry_index,
    exit_engine
):
    """
    Search candles after entry and use the configured
    ExitEngine to determine the first exit.

    Returns:
        exit_index
        exit_price
        exit_reason
    """

    entry_row = symbol_df.loc[entry_index]

    entry_price = entry_row["close"]

    # --------------------------------------------------------
    # USER'S SL LOGIC
    # --------------------------------------------------------

    sl = min(
        entry_row["low"],
        entry_row["9_ema"]
    )

    # --------------------------------------------------------
    # TARGET
    # --------------------------------------------------------

    target = (
        entry_price *
        (1 + TARGET_PERCENT)
    )

    # --------------------------------------------------------
    # FUTURE CANDLES
    # --------------------------------------------------------

    future_rows = symbol_df.loc[
        symbol_df.index > entry_index
    ]

    for idx, row in future_rows.iterrows():

        result = exit_engine.check(
            row=row,
            entry_price=entry_price,
            sl=sl,
            target=target
        )

        if result is not None:

            return (
                idx,
                result.exit_price,
                result.exit_reason
            )

    return None, None, None


# ============================================================
# GENERATE TRADES
# ============================================================

trades = []


for symbol, symbol_df in df.groupby(
    "symbol",
    sort=False
):

    symbol_df = (
        symbol_df
        .sort_values("date")
        .copy()
    )

    position_open = False
    exit_index = None

    # --------------------------------------------------------
    # LOOP THROUGH STOCK DATA
    # --------------------------------------------------------

    for idx, row in symbol_df.iterrows():

        # ----------------------------------------------------
        # POSITION ALREADY OPEN
        # ----------------------------------------------------

        if position_open:

            if idx <= exit_index:
                continue

            position_open = False
            exit_index = None

        # ----------------------------------------------------
        # NO SIGNAL
        # ----------------------------------------------------

        if not row["trade_signal"]:
            continue

        # ----------------------------------------------------
        # ENTRY
        # ----------------------------------------------------

        entry_date = row["date"]

        entry_price = row["close"]

        # ----------------------------------------------------
        # STOP LOSS
        # ----------------------------------------------------

        sl = min(
            row["low"],
            row["9_ema"]
        )

        # ----------------------------------------------------
        # TARGET
        # ----------------------------------------------------

        target = (
            entry_price *
            (1 + TARGET_PERCENT)
        )

        # ----------------------------------------------------
        # FIND EXIT
        # ----------------------------------------------------

        (
            exit_idx,
            exit_price,
            exit_reason
        ) = find_exit(
            symbol_df=symbol_df,
            entry_index=idx,
            exit_engine=exit_engine
        )

        # ----------------------------------------------------
        # NO EXIT
        # ----------------------------------------------------

        if exit_idx is None:

            position_open = True

            # Don't add incomplete trade
            continue

        # ----------------------------------------------------
        # EXIT DATE
        # ----------------------------------------------------

        exit_row = symbol_df.loc[exit_idx]

        exit_date = exit_row["date"]

        # ----------------------------------------------------
        # PNL
        # ----------------------------------------------------

        pnl = (
            exit_price -
            entry_price
        )

        pnl_percent = (
            (
                exit_price -
                entry_price
            ) /
            entry_price
        ) * 100

        # ----------------------------------------------------
        # HOLDING PERIOD
        # ----------------------------------------------------

        holding_period = (
            exit_date -
            entry_date
        ).days

        # ----------------------------------------------------
        # STORE TRADE
        # ----------------------------------------------------

        trades.append({

            "symbol": symbol,

            "entry_date": entry_date,
            "exit_date": exit_date,

            "entry": entry_price,

            "sl": sl,

            "target": target,

            "exit_price": exit_price,

            "exit_reason": exit_reason,

            "holding_period": holding_period,

            "pnl": pnl,

            "pnl_percent": pnl_percent,

            "entry_index": idx,

            "exit_index": exit_idx
        })

        # ----------------------------------------------------
        # POSITION IS OPEN UNTIL EXIT
        # ----------------------------------------------------

        position_open = True

        exit_index = exit_idx


# ============================================================
# CREATE TRADE DATAFRAME
# ============================================================

trades_df = pd.DataFrame(trades)


# ============================================================
# SORT TRADES
# ============================================================

if not trades_df.empty:

    trades_df = (
        trades_df
        .sort_values("entry_date")
        .reset_index(drop=True)
    )


# ============================================================
# DISPLAY
# ============================================================

print(
    "\n========== BACKTEST TRADES ==========\n"
)

if trades_df.empty:

    print("No completed trades found.")

else:

    print(
        trades_df.to_string(
            index=False
        )
    )


# ============================================================
# SAVE
# ============================================================

trades_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    f"\nSaved to:\n{OUTPUT_PATH}"
)


========== BACKTEST TRADES ==========

 symbol entry_date  exit_date  entry     sl  target  exit_price exit_reason  holding_period     pnl  pnl_percent  entry_index  exit_index
    BEL 2020-01-02 2020-01-06  34.38  33.30  37.818      33.300          SL               4  -1.080    -3.141361            1           3
   VEDL 2020-01-02 2020-01-06  59.72  58.16  65.692      58.160          SL               4  -1.560    -2.612190         4606        4608
    BEL 2020-01-09 2020-01-10  32.33  31.93  35.563      32.400       9_EMA               1   0.070     0.216517            6           7
   NTPC 2020-01-13 2020-01-16 121.75 120.10 133.925     121.450       9_EMA               3  -0.300    -0.246407         2948        2951
    BEL 2020-01-14 2020-01-22  33.28  32.65  36.608      33.650       9_EMA               8   0.370     1.111779            9          15
   VEDL 2020-01-14 2020-01-17  61.80  59.57  67.980      59.570          SL               3  -2.230    -3.608414         4614       

In [2]:
import pandas as pd
import numpy as np


def backtest_report(
    df: pd.DataFrame,
    capital: float
) -> dict:
    """
    Generate a performance report from completed trades.

    Required columns
    ----------------
    entry_date
        Date on which the trade was entered.

    exit_date
        Date on which the trade was exited.

    entry
        Entry price.

    exit_price
        Exit price.

    pnl
        Absolute P&L of the trade.

    pnl_percent
        Percentage return of the trade.

    exit_reason
        Reason for exit, e.g. TARGET, SL, 9 EMA.

    Optional columns
    ----------------
    symbol
        Stock/instrument name.

    capital
        Starting capital supplied separately.

    Returns
    -------
    dict
        Backtest performance metrics.
    """

    # ========================================================
    # VALIDATION
    # ========================================================

    required_columns = {
        "entry_date",
        "exit_date",
        "entry",
        "exit_price",
        "pnl",
        "pnl_percent",
        "exit_reason",
    }

    missing_columns = (
        required_columns - set(df.columns)
    )

    if missing_columns:
        raise ValueError(
            f"Missing required columns: "
            f"{sorted(missing_columns)}"
        )

    if capital <= 0:
        raise ValueError(
            "Capital must be greater than 0."
        )


    # ========================================================
    # COPY DATA
    # ========================================================

    trades = df.copy()


    # ========================================================
    # DATE CONVERSION
    # ========================================================

    trades["entry_date"] = pd.to_datetime(
        trades["entry_date"]
    )

    trades["exit_date"] = pd.to_datetime(
        trades["exit_date"]
    )


    # ========================================================
    # REMOVE OPEN / INVALID TRADES
    # ========================================================

    trades = trades[
        trades["exit_price"].notna()
    ].copy()

    trades = trades[
        trades["exit_date"].notna()
    ].copy()


    if trades.empty:

        return {
            "initial_capital": capital,
            "ending_capital": capital,

            "total_trades": 0,
            "winning_trades": 0,
            "losing_trades": 0,

            "total_pnl": 0,
            "average_pnl": 0,

            "win_rate": 0,

            "profit_factor": 0,

            "max_drawdown": 0,
            "max_drawdown_percent": 0,

            "winning_streak": 0,
            "losing_streak": 0,

            "cagr": 0,
            "sharpe_ratio": 0,
        }


    # ========================================================
    # SORT TRADES
    # ========================================================

    trades = (
        trades
        .sort_values("exit_date")
        .reset_index(drop=True)
    )


    # ========================================================
    # BASIC TRADE STATISTICS
    # ========================================================

    total_trades = len(trades)

    winning_trades = (
        trades["pnl"] > 0
    ).sum()

    losing_trades = (
        trades["pnl"] <= 0
    ).sum()


    # ========================================================
    # TOTAL P&L
    # ========================================================

    total_pnl = trades["pnl"].sum()

    average_pnl = trades["pnl"].mean()


    # ========================================================
    # WIN RATE
    # ========================================================

    win_rate = (
        winning_trades
        / total_trades
    ) * 100


    # ========================================================
    # PROFIT FACTOR
    # ========================================================

    gross_profit = trades.loc[
        trades["pnl"] > 0,
        "pnl"
    ].sum()

    gross_loss = abs(
        trades.loc[
            trades["pnl"] < 0,
            "pnl"
        ].sum()
    )


    if gross_loss > 0:

        profit_factor = (
            gross_profit
            / gross_loss
        )

    else:

        profit_factor = np.inf


    # ========================================================
    # EQUITY CURVE
    # ========================================================

    trades["equity"] = (
        capital
        + trades["pnl"].cumsum()
    )


    # ========================================================
    # MAX DRAWDOWN
    # ========================================================

    trades["peak"] = (
        trades["equity"]
        .cummax()
    )

    trades["drawdown"] = (
        trades["equity"]
        - trades["peak"]
    )

    trades["drawdown_percent"] = (
        trades["drawdown"]
        / trades["peak"]
    ) * 100


    max_drawdown = (
        trades["drawdown"].min()
    )

    max_drawdown_percent = (
        trades["drawdown_percent"].min()
    )


    # ========================================================
    # WINNING STREAK
    # ========================================================

    winning_streak = 0
    current_winning_streak = 0

    for pnl in trades["pnl"]:

        if pnl > 0:

            current_winning_streak += 1

            winning_streak = max(
                winning_streak,
                current_winning_streak
            )

        else:

            current_winning_streak = 0


    # ========================================================
    # LOSING STREAK
    # ========================================================

    losing_streak = 0
    current_losing_streak = 0

    for pnl in trades["pnl"]:

        if pnl < 0:

            current_losing_streak += 1

            losing_streak = max(
                losing_streak,
                current_losing_streak
            )

        else:

            current_losing_streak = 0


    # ========================================================
    # CAGR
    # ========================================================

    start_date = trades["entry_date"].min()

    end_date = trades["exit_date"].max()

    total_days = (
        end_date - start_date
    ).days


    ending_capital = (
        capital + total_pnl
    )


    if (
        total_days > 0
        and ending_capital > 0
    ):

        years = total_days / 365.25

        cagr = (
            (
                ending_capital
                / capital
            ) ** (1 / years)
            - 1
        ) * 100

    else:

        cagr = 0


    # ========================================================
    # SHARPE RATIO
    # ========================================================

    # Trade-level Sharpe ratio.
    #
    # This assumes zero risk-free rate.
    #
    # NOTE:
    # This is NOT the same as calculating Sharpe
    # from daily portfolio returns.

    returns = (
        trades["pnl"]
        / capital
    )


    if (
        returns.std(ddof=1) != 0
        and len(returns) > 1
    ):

        sharpe_ratio = (
            returns.mean()
            / returns.std(ddof=1)
        ) * np.sqrt(len(returns))

    else:

        sharpe_ratio = 0


    # ========================================================
    # AVERAGE WIN / LOSS
    # ========================================================

    average_win = trades.loc[
        trades["pnl"] > 0,
        "pnl"
    ].mean()

    average_loss = trades.loc[
        trades["pnl"] < 0,
        "pnl"
    ].mean()


    # ========================================================
    # RETURN REPORT
    # ========================================================

    report = {

        "initial_capital":
            round(capital, 2),

        "ending_capital":
            round(ending_capital, 2),

        "total_pnl":
            round(total_pnl, 2),

        "average_pnl":
            round(average_pnl, 2),

        "total_trades":
            int(total_trades),

        "winning_trades":
            int(winning_trades),

        "losing_trades":
            int(losing_trades),

        "win_rate":
            round(win_rate, 2),

        "average_win":
            round(average_win, 2),

        "average_loss":
            round(average_loss, 2),

        "profit_factor":
            round(profit_factor, 2)
            if np.isfinite(profit_factor)
            else float("inf"),

        "max_drawdown":
            round(max_drawdown, 2),

        "max_drawdown_percent":
            round(max_drawdown_percent, 2),

        "winning_streak":
            int(winning_streak),

        "losing_streak":
            int(losing_streak),

        "cagr":
            round(cagr, 2),

        "sharpe_ratio":
            round(sharpe_ratio, 2),

    }


    return report

In [3]:
backtest_report(trades_df,capital=10000)

{'initial_capital': 10000,
 'ending_capital': np.float64(10390.88),
 'total_pnl': np.float64(390.88),
 'average_pnl': np.float64(0.69),
 'total_trades': 570,
 'winning_trades': 222,
 'losing_trades': 348,
 'win_rate': np.float64(38.95),
 'average_win': np.float64(7.74),
 'average_loss': np.float64(-3.81),
 'profit_factor': np.float64(1.29),
 'max_drawdown': np.float64(-168.59),
 'max_drawdown_percent': np.float64(-1.61),
 'winning_streak': 5,
 'losing_streak': 9,
 'cagr': np.float64(0.58),
 'sharpe_ratio': np.float64(1.99)}